# 🏥 Medical RAG Chatbot — MedQuAD Dataset
### Internship Task 3 (Real-World) | LlamaIndex + HuggingFace + Gradio

---

**What this notebook does:**
- 📂 Loads your `medquad.csv` directly from Google Drive
- 🔍 Auto-detects columns and builds a real medical knowledge base
- 🧠 Embeds all Q&A pairs using HuggingFace sentence-transformers
- 💬 Launches a beautiful **Gradio Chat UI** to ask medical questions
- 📚 Shows source citations for every answer

**Stack:** LlamaIndex 0.12 · HuggingFace · Gradio  
**Dataset:** MedQuAD (Medical Question Answering Dataset)

---

## ⚠️ IMPORTANT — Run This First!
After Cell 1 installs packages, **Runtime → Restart session**, then run from Cell 2 onwards.

## ✅ Cell 1 — Install Packages (Then Restart Runtime!)

In [ ]:
# ── Step 1: Clean slate ───────────────────────────────────────────────────
!pip uninstall llama-index llama-index-core llama-index-embeddings-huggingface \
               llama-index-llms-huggingface llama-index-embeddings-openai \
               llama-index-readers-file -y -q

# ── Step 2: Install llama-index (let it pick compatible versions) ─────────
!pip install llama-index -q
!pip install llama-index-embeddings-huggingface -q
!pip install llama-index-llms-huggingface -q

# ── Step 3: Other dependencies ────────────────────────────────────────────
!pip install "sentence-transformers>=2.7.0" -q
!pip install "transformers>=4.40.0" accelerate -q
!pip install gradio -q

# ── Step 4: Verify llama-index-core actually installed ────────────────────
import importlib.metadata as meta
print(f"\nllama-index-core: {meta.version('llama-index-core')}")
print("\n✅ Done! Now: Runtime → Restart session, then run Cell 2 onwards.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 115.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 85.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.2/121.2 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 14.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 108.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.9 MB/s eta 0:00:00

llama-index-core: 0.14.22

✅ Done! Now: Runtime →

## ✅ Cell 2 — Import Libraries

In [ ]:
# ── Verify versions are correct ───────────────────────────────────────────
import importlib.metadata as meta
for pkg in ['llama-index-core', 'sentence-transformers', 'gradio']:
    try:
        print(f'  {pkg}: {meta.version(pkg)}')
    except:
        print(f'  {pkg}: NOT FOUND — rerun Cell 1')

import warnings, os, torch, pandas as pd
warnings.filterwarnings('ignore')

from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import SimilarityPostprocessor
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.huggingface import HuggingFaceLLM
from transformers import AutoTokenizer
import gradio as gr

print(f'\n✅ All imports successful!')
print(f'   GPU available : {torch.cuda.is_available()}')
print(f'   Device        : {"GPU " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

  llama-index-core: 0.14.22
  sentence-transformers: 5.4.1
  gradio: 5.50.0

✅ All imports successful!
   GPU available : True
   Device        : GPU Tesla T4


## ✅ Cell 3 — Mount Google Drive & Load MedQuAD CSV

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Auto-search for medquad.csv in common locations ───────────────────────
SEARCH_PATHS = [
    '/content/drive/MyDrive/medquad.csv',
    '/content/drive/MyDrive/datasets/medquad.csv',
    '/content/drive/MyDrive/data/medquad.csv',
    '/content/drive/MyDrive/internship/medquad.csv',
    '/content/drive/MyDrive/Colab Notebooks/medquad.csv',
]

csv_path = None
for p in SEARCH_PATHS:
    if os.path.exists(p):
        csv_path = p
        break

if csv_path is None:
    # ── Manual fallback ───────────────────────────────────────────────────
    # If auto-search fails, set your path here:
    csv_path = '/content/drive/MyDrive/medquad.csv'
    print(f'⚠️  Could not auto-find CSV.')
    print(f'   Using default path: {csv_path}')
    print(f'   If wrong, update csv_path above manually.')
else:
    print(f'✅ Found CSV: {csv_path}')

# ── Load CSV ──────────────────────────────────────────────────────────────
df_raw = pd.read_csv(csv_path)
print(f'\n📊 Dataset loaded!')
print(f'   Shape   : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
print(f'   Columns : {list(df_raw.columns)}')
print(f'\n🔍 Preview (first 2 rows):')
df_raw.head(2)

Mounted at /content/drive
✅ Found CSV: /content/drive/MyDrive/medquad.csv

📊 Dataset loaded!
   Shape   : 16,412 rows × 4 columns
   Columns : ['question', 'answer', 'source', 'focus_area']

🔍 Preview (first 2 rows):


,question,answer,source,focus_area
0,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma
1,What causes Glaucoma ?,"Nearly 2.7 million people have glaucoma, a lea...",NIHSeniorHealth,Glaucoma


## ✅ Cell 4 — Auto-Detect Columns & Clean Dataset

In [ ]:
# ── Auto-detect column names ──────────────────────────────────────────────
def find_col(df, keywords):
    for col in df.columns:
        if any(k in col.lower() for k in keywords):
            return col
    return None

question_col = find_col(df_raw, ['question', 'query', 'input', 'q'])
answer_col   = find_col(df_raw, ['answer', 'response', 'output', 'text', 'content', 'a'])
topic_col    = find_col(df_raw, ['topic', 'focus', 'disease', 'category', 'source', 'type'])

print('🔍 Auto-detected columns:')
print(f'   Question : {question_col}')
print(f'   Answer   : {answer_col}')
print(f'   Topic    : {topic_col} (optional)')

# ── If auto-detection wrong, override manually here ───────────────────────
# question_col = 'question'
# answer_col   = 'answer'
# topic_col    = 'focus_area'

assert question_col, '❌ Question column not found. Set question_col manually.'
assert answer_col,   '❌ Answer column not found. Set answer_col manually.'

# ── Select and clean relevant columns ─────────────────────────────────────
cols = [c for c in [question_col, answer_col, topic_col] if c]
df = df_raw[cols].copy().dropna(subset=[question_col, answer_col])
df[question_col] = df[question_col].astype(str).str.strip()
df[answer_col]   = df[answer_col].astype(str).str.strip()

# Remove very short / empty answers
df = df[df[answer_col].str.len() > 30].reset_index(drop=True)

# ── Sample for Colab free tier ────────────────────────────────────────────
# Increase MAX_ROWS if you have Colab Pro or GPU
MAX_ROWS = 2000
if len(df) > MAX_ROWS:
    df = df.sample(n=MAX_ROWS, random_state=42).reset_index(drop=True)
    print(f'\n⚠️  Sampled to {MAX_ROWS} rows (Colab free tier limit).')
    print(f'   Increase MAX_ROWS for more coverage.')

print(f'\n✅ Clean dataset ready: {len(df):,} Q&A pairs')
print(f'\n📄 Sample:')
print(f'   Q: {df[question_col].iloc[0]}')
print(f'   A: {df[answer_col].iloc[0][:200]}...')

🔍 Auto-detected columns:
   Question : question
   Answer   : answer
   Topic    : source (optional)

⚠️  Sampled to 2000 rows (Colab free tier limit).
   Increase MAX_ROWS for more coverage.

✅ Clean dataset ready: 2,000 Q&A pairs

📄 Sample:
   Q: What are the symptoms of Juvenile Huntington disease ?
   A: What are the signs and symptoms of Juvenile Huntington disease? A common sign of juvenile HD is a rapid decline in school performance. Symptoms can also include subtle changes in handwriting and sligh...


## ✅ Cell 5 — Build LlamaIndex Documents & Chunks

In [ ]:
# ── Convert each CSV row to a LlamaIndex Document ─────────────────────────
# Format: "Question: ...\'nAnswer: ..." so LLM understands both parts

documents = []
for _, row in df.iterrows():
    topic = str(row[topic_col]).strip() if topic_col and topic_col in row else 'Medical'
    text  = f"Question: {row[question_col]}\nAnswer: {row[answer_col]}"
    documents.append(Document(
        text=text,
        metadata={
            'question' : str(row[question_col])[:120],
            'topic'    : topic,
            'source'   : 'MedQuAD'
        }
    ))

# ── Split into chunks ─────────────────────────────────────────────────────
splitter = SentenceSplitter(chunk_size=300, chunk_overlap=40)
nodes    = splitter.get_nodes_from_documents(documents)

print(f'✅ Documents  : {len(documents):,}')
print(f'   Chunks     : {len(nodes):,}')
print(f'   Chunk size : 300 tokens | Overlap: 40')
print(f'\n📄 Sample chunk preview:')
print(nodes[0].text[:300])

✅ Documents  : 2,000
   Chunks     : 3,369
   Chunk size : 300 tokens | Overlap: 40

📄 Sample chunk preview:
Question: What are the symptoms of Juvenile Huntington disease ?
Answer: What are the signs and symptoms of Juvenile Huntington disease? A common sign of juvenile HD is a rapid decline in school performance. Symptoms can also include subtle changes in handwriting and slight problems with movement, s


## ✅ Cell 6 — Load Embedding Model

In [ ]:
# ── HuggingFace Embedding Model ───────────────────────────────────────────
# all-MiniLM-L6-v2 : 384-dim, fast, free, great semantic search quality
# Downloads ~80MB from HuggingFace Hub on first run

print('⏳ Loading embedding model...')

embed_model = HuggingFaceEmbedding(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    device='cuda' if torch.cuda.is_available() else 'cpu'
)
Settings.embed_model = embed_model

# Quick test
test = embed_model.get_text_embedding('diabetes symptoms')
print(f'✅ Embedding model ready!')
print(f'   Model      : sentence-transformers/all-MiniLM-L6-v2')
print(f'   Dimensions : {len(test)}')
print(f'   Device     : {"GPU" if torch.cuda.is_available() else "CPU"}')

⏳ Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model ready!
   Model      : sentence-transformers/all-MiniLM-L6-v2
   Dimensions : 384
   Device     : GPU


## ✅ Cell 7 — Load HuggingFace LLM

In [ ]:
from transformers import T5ForConditionalGeneration, T5Tokenizer
from llama_index.core.llms import CustomLLM, CompletionResponse, LLMMetadata
from llama_index.core.llms.callbacks import llm_completion_callback
from llama_index.core import Settings
from typing import Any, Generator
import torch

#MODEL_NAME = 'google/flan-t5-base'
MODEL_NAME = 'google/flan-t5-large'   # was: flan-t5-base
print(f'⏳ Loading LLM: {MODEL_NAME} ...')

# Load model directly (bypasses HuggingFaceLLM's CausalLM assumption)
_tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME, legacy=False)
_device    = 'cuda' if torch.cuda.is_available() else 'cpu'
_model     = T5ForConditionalGeneration.from_pretrained(MODEL_NAME).to(_device)
_model.eval()

print(f'   Model loaded on: {_device.upper()}')

# ── Custom LlamaIndex-compatible wrapper ──────────────────────────────────
class FlanT5LLM(CustomLLM):
    context_window: int = 2048
    num_output: int = 300
    model_name: str = MODEL_NAME

    @property
    def metadata(self) -> LLMMetadata:
        return LLMMetadata(
            context_window=self.context_window,
            num_output=self.num_output,
            model_name=self.model_name,
        )

    @llm_completion_callback()
    def complete(self, prompt: str, **kwargs) -> CompletionResponse:
        inputs = _tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=512
        ).to(_device)
        with torch.no_grad():
            outputs = _model.generate(
                **inputs,
                max_new_tokens=300,
                do_sample=False   # greedy = more factual for medical Q&A
            )
        text = _tokenizer.decode(outputs[0], skip_special_tokens=True)
        return CompletionResponse(text=text)

    @llm_completion_callback()
    def stream_complete(self, prompt: str, **kwargs) -> Generator:
        response = self.complete(prompt)
        yield CompletionResponse(text=response.text, delta=response.text)

llm = FlanT5LLM()
Settings.llm = llm

print(f'✅ LLM ready!')
print(f'   Model  : {MODEL_NAME}')
print(f'   Device : {_device.upper()}')
print(f'   Type   : Seq2Seq (T5ForConditionalGeneration)')

⏳ Loading LLM: google/flan-t5-large ...


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

   Model loaded on: CUDA
✅ LLM ready!
   Model  : google/flan-t5-large
   Device : CUDA
   Type   : Seq2Seq (T5ForConditionalGeneration)


## ✅ Cell 8 — Build Vector Index

In [ ]:
# ── Build Vector Index ────────────────────────────────────────────────────
# Embeds all chunks → stores vectors for fast similarity search
# This is the most time-consuming step (~1-3 min for 2000 rows)

print('⏳ Building vector index...')
print(f'   Embedding {len(nodes):,} chunks — please wait...')

index = VectorStoreIndex(
    nodes,
    embed_model=embed_model,
    show_progress=True
)

# ── Build Query Engine ────────────────────────────────────────────────────
retriever    = VectorIndexRetriever(index=index, similarity_top_k=4)
postproc     = SimilarityPostprocessor(similarity_cutoff=0.35)
query_engine = RetrieverQueryEngine(
    retriever=retriever,
    node_postprocessors=[postproc]
)

print(f'\n✅ Vector index built!')
print(f'   Vectors indexed  : {len(nodes):,}')
print(f'   Top-K retrieval  : 4 chunks')
print(f'   Similarity cutoff: 0.35')
print(f'\n🚀 RAG pipeline ready — proceed to Cell 9 to launch Gradio!')

⏳ Building vector index...
   Embedding 3,369 chunks — please wait...


Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/1321 [00:00<?, ?it/s]


✅ Vector index built!
   Vectors indexed  : 3,369
   Top-K retrieval  : 4 chunks
   Similarity cutoff: 0.35

🚀 RAG pipeline ready — proceed to Cell 9 to launch Gradio!


## ✅ Cell 9 — Launch Gradio Chat Interface 🚀

> After running this cell, click the **public `gradio.live` URL** printed below to open your chatbot!

In [ ]:
import gradio as gr

CSS = """
@import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@300;400;500;600&family=DM+Serif+Display&display=swap');

body, .gradio-container { font-family: 'DM Sans', sans-serif !important; background: #f8f9fb !important; }

.chat-window { background: #fff !important; border: 1px solid #eaedf2 !important; border-radius: 16px !important; }
.message-bubble-border.bot { background: #fff !important; border: 1px solid #eaedf2 !important; box-shadow: 0 1px 4px rgba(0,0,0,.05) !important; border-radius: 14px !important; }
.message-bubble-border.user { background: #0f1923 !important; color: #e8f0f6 !important; border-radius: 14px !important; }

.send-btn { background: #0f1923 !important; border-radius: 10px !important; }
.send-btn:hover { background: #1a9e75 !important; }

footer { display: none !important; }
"""

HEADER_HTML = f"""
<div style="background:#0f1923;padding:20px 28px;border-radius:16px 16px 0 0;display:flex;align-items:center;justify-content:space-between;margin-bottom:16px">
  <div style="display:flex;align-items:center;gap:14px">
    <div style="width:42px;height:42px;background:linear-gradient(135deg,#1a9e75,#0d6e56);border-radius:11px;display:flex;align-items:center;justify-content:center;font-size:20px">🏥</div>
    <div>
      <div style="font-size:17px;font-weight:600;color:#fff;letter-spacing:-0.3px">MedRAG</div>
      <div style="font-size:11px;color:#4a6070">Medical Q&A · MedQuAD · LlamaIndex</div>
    </div>
  </div>
  <div style="display:flex;gap:20px;text-align:center">
    <div><div style="font-size:16px;font-weight:600;color:#1a9e75">{len(df):,}</div><div style="font-size:10px;color:#4a6070">Q&A Pairs</div></div>
    <div><div style="font-size:16px;font-weight:600;color:#1a9e75">{len(nodes):,}</div><div style="font-size:10px;color:#4a6070">Chunks</div></div>
    <div><div style="font-size:16px;font-weight:600;color:#1a9e75">384</div><div style="font-size:10px;color:#4a6070">Embed Dims</div></div>
  </div>
</div>
"""

SAMPLES = [
    "What are the symptoms of diabetes?",
    "How is hypertension treated?",
    "What causes asthma attacks?",
    "What are the warning signs of a heart attack?",
    "How is tuberculosis diagnosed?",
    "What is the treatment for malaria?",
]

def rag_respond(user_msg, history):
    if not user_msg.strip():
        return history, ""

    try:
        retrieved_nodes = retriever.retrieve(user_msg)

        if not retrieved_nodes:
            answer = "No relevant information found in the medical database for your question."
        else:
            best_node = retrieved_nodes[0]
            raw_text  = best_node.text.strip()
            answer    = raw_text.split("Answer:", 1)[1].strip() if "Answer:" in raw_text else raw_text

        # Build clean source citations
        if retrieved_nodes:
            answer += "\n\n**Sources from MedQuAD:**"
            for i, n in enumerate(retrieved_nodes[:3]):
                topic = n.metadata.get("topic", "Medical")
                q_pre = n.metadata.get("question", "")[:70]
                score = n.score
                answer += f"\n`[{i+1}]` **{topic}** — \"{q_pre}...\" *(similarity: {score:.3f})*"

    except Exception as e:
        answer = f"Error processing your question: {str(e)}"

    history.append({"role": "user",      "content": user_msg})
    history.append({"role": "assistant", "content": answer})
    return history, ""


def clear_history():
    return [], ""


with gr.Blocks(css=CSS, title="MedRAG — Medical Q&A") as demo:

    gr.HTML(HEADER_HTML)

    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
                label="",
                height=460,
                type="messages",
                bubble_full_width=False,
                avatar_images=("👤", "🏥"),
                placeholder='<div style="text-align:center;padding:40px;color:#8a93a3"><div style="font-size:40px;margin-bottom:12px">🏥</div><div style="font-size:17px;font-weight:600;color:#2a3040;font-family:serif">Ask a Medical Question</div><div style="font-size:13px;margin-top:6px">Search through curated MedQuAD Q&A pairs</div></div>',
                elem_classes=["chat-window"]
            )
            with gr.Row():
                msg = gr.Textbox(
                    placeholder="e.g. What are the symptoms of malaria?",
                    label="",
                    lines=2,
                    scale=5,
                    container=False
                )
                with gr.Column(scale=1, min_width=110):
                    send_btn  = gr.Button("Send ➤",  variant="primary",   size="lg",  elem_classes=["send-btn"])
                    clear_btn = gr.Button("Clear",   variant="secondary", size="sm")

        with gr.Column(scale=1, min_width=210):
            gr.Markdown("#### 💡 Quick Questions")
            for s in SAMPLES:
                gr.Button(s, size="sm").click(fn=lambda x=s: x, outputs=msg)

            gr.Markdown("---")
            gr.Markdown("""
#### ℹ️ How It Works
1. Your question is **embedded** into a vector
2. Top **4 chunks** retrieved from MedQuAD
3. Best matching **answer returned directly**
4. **Sources cited** with similarity scores
            """)

    state = gr.State([])
    send_btn.click(fn=rag_respond,    inputs=[msg, state], outputs=[chatbot, msg])
    msg.submit(fn=rag_respond,        inputs=[msg, state], outputs=[chatbot, msg])
    clear_btn.click(fn=clear_history,                      outputs=[chatbot, msg])

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fc54a398798fabea74.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
## 📋 Pipeline Summary

| Component | Detail |
|---|---|
| **Dataset** | medquad.csv from Google Drive |
| **Column Detection** | Automatic |
| **Framework** | LlamaIndex 0.12 |
| **Embedding** | sentence-transformers/all-MiniLM-L6-v2 |
| **LLM** | google/flan-t5-base (free, no API key) |
| **Chunk Size** | 300 tokens / 40 overlap |
| **Top-K** | 4 chunks per query |
| **UI** | Gradio Chat (public URL) |

### 🚀 Run Order
1. **Cell 1** → install packages → **Restart Runtime**
2. **Cell 2 → 8** → run all normally
3. **Cell 9** → Gradio UI launches, click the `gradio.live` URL

### 🛠️ Common Fixes
| Problem | Fix |
|---|---|
| `ModuleNotFoundError` | Restart runtime after Cell 1 |
| CSV not found | Update `csv_path` manually in Cell 3 |
| Column not detected | Uncomment manual overrides in Cell 4 |
| Out of memory | Reduce `MAX_ROWS` in Cell 4 |
| Slow answers | Switch to GPU runtime in Colab settings |